# Framework + R0 Baseline

## Step 1: How the Original Repair Loop Works

**File:** `chinatravel/agent/nesy_verifier/llm_modulo.py`

Main method: `LLMModuloAgent.solve(problem, prob_idx, oracle_verifier)`

The original R0 does not have a separate repair function. It repairs an itinerary by prompting the LLM again with the previous plan and verifier feedback.

**Flow:**

1. **Collect information:** Shortlist transport, hotels, attractions and restaurants from the ChinaTravel environment.
2. **Generate:** Use `INITIAL_PROMPT` to generate the initial itinerary (Step 0).
3. **Verify:** Run `collect_commonsense_constraints_error` and `collect_personal_error` to identify violations.
4. **Repair:** If errors are reported, use `BACK_PROMPT` to generate a new itinerary using the previous plan and feedback.
5. **Repeat:** Continue until no errors are reported or the repair limit is reached.
6. **Return:** Output the final itinerary and verification results.

**Key points:**

- `max_steps` is the maximum number of repair attempts, controlled by `--refine_steps` (default 10).
- R0 regenerates the entire itinerary during repair. It does not protect previously satisfied constraints.
- The original verifier returns `error_info`, a list of reported violations. An empty list means no errors were reported, but some checks may have been skipped.
- Returning `True` from `solve()` does not necessarily mean all constraints passed.

---

## Step 2: Setting Up and Testing R0

### 2.1 Environment Setup

Set up ChinaTravel using Python 3.12.8 and the dependencies in `requirements.txt`.

**Configuration:**

- **Dataset:** ChinaTravel dataset from Hugging Face
- **Sandbox:** ChinaTravel-Sandbox (Chinese)
- **LLM:** GPT-4.1 through OpenRouter
- **Baseline:** LLM-Modulo (R0)

**Issues encountered and fixes:**

1. **Dependencies:** Used NumPy 1.26.4 and PyArrow 19.0.1. Fixed the missing `_lzma` module by reinstalling Python with XZ support.
2. **Dataset format:** `hard_logic_py` was stored as a string instead of a list. Updated `load_datasets.py` to convert it using `ast.literal_eval()`.
3. **Missing sandbox:** Downloaded the official ChinaTravel-Sandbox into `chinatravel/environment/database/`.
4. **Missing import:** Added `import sys` to `llm_modulo.py`.
5. **OpenRouter:** Updated the model configuration to support `openai/gpt-4.1`.

I also added `scripts/setup_project.py` to automate sandbox installation and `.env.example` for API configuration.

Teammates can follow the setup instructions in `README.md` instead of repeating these steps manually.

### 2.2 Testing the Original R0

Ran the original R0 using `--refine_steps 2`.

**Test 1 — Easy split**

Case: `e20241028160248698752`

- One-day trip from Shanghai to Hangzhou with a budget of 1,500 RMB.
- The initial itinerary passed the reported checks.
- No repair was needed (`ITER 0 ERROR: []`).

**Test 2 — Medium split**

Case: `e20241028160845920703`

- Two-day trip from Shenzhen to Guangzhou, budget 1,000 RMB, hotpot preference and single-bed room.
- The initial itinerary violated a dinner-timing constraint.
- R0 regenerated the itinerary using verifier feedback.
- After one repair, the original verifier returned `ITER 1 ERROR: []`.

This confirmed that the original repair loop could respond to the observed timing violation. However, personal constraints were not fully checked because `hard_logic_nl` was missing.

### 2.3 Logging

The original ChinaTravel implementation saves files for each iteration:

| File | Purpose |
|---|---|
| `step_0_prompt.txt` | Initial generation prompt |
| `step_0_plan.json` | Initial itinerary and verification results |
| `step_0_error.err` | Initial verifier feedback |
| `step_1_...` | Corresponding files after repair |

The saved plans also contain token usage and execution time.

The original logs do not directly track constraint transitions. I added this to `shared_repair_loop.py`, which records which personal constraints were fixed, newly violated, still failed or still satisfied.

---

## Step 3: Shared Framework Implementation

I separated the repair process from the original ChinaTravel agent so R0–R4 can share the same planner, verifier and repair loop.

### 3.1 Files Added

All files are located in `chinatravel/agent/nesy_verifier/`.

| File | Purpose |
|---|---|
| `shared_verifier.py` | Checks commonsense and personal constraints |
| `shared_repair_loop.py` | Handles repair attempts, verification, acceptance/rejection and logging |
| `repair_strategies/base.py` | Defines the common interface for R0–R4 |
| `repair_strategies/r0.py` | Implements R0 whole-plan regeneration |

I also updated `run_exp.py`, `load_model.py` and `llm_modulo.py` to support selecting repair strategies.

### 3.2 How the Shared Framework Works

1. ChinaTravel generates the initial itinerary.
2. The shared verifier checks commonsense and personal constraints.
3. The selected repair strategy proposes a revised itinerary.
4. The shared verifier checks the proposed repair.
5. The strategy decides whether to accept or reject it.
6. Repeat until verification passes or the repair limit is reached.

The original ChinaTravel R0 loop remains available when no repair strategy is selected.

### 3.3 Improvements Added

- **Shared verification:** Uses `hard_logic_py` directly to check personal constraints, even when `hard_logic_nl` is missing.
- **Raw itinerary preservation:** Keeps the original itinerary separate from the translated transport version.
- **Acceptance/rejection:** Supports accepting or rejecting repair proposals and keeping the previous plan when a proposal is rejected. The decision is left to each strategy.
- **Constraint-transition logging:** Records which personal constraints were fixed, newly violated, still failed, still passed or could not be evaluated.
- **Verifier reliability:** Separates genuine constraint violations from technical errors, including verifier exceptions, transport translation failures and invalid itinerary structures.
- **Separate outputs:** Different repair strategies have separate result and log directories.

The shared framework does not automatically repair invalid plans, protect constraints or reject harmful changes. These decisions belong to the individual R0–R4 strategies.

### 3.4 Testing

Completed the following:

- Tested the original R0 on easy and medium ChinaTravel cases.
- Tested the shared verifier on a saved itinerary.
- Tested standalone R0 using a mock LLM.
- Ran the new R0 end-to-end on a medium case.
- Passed all 7 shared-framework unit tests.

**New R0 end-to-end test:**

| Iteration | Reported errors | Personal constraints |
|---|---:|---|
| Initial plan | 38 | 8/8 passed |
| Repair 1 | 3 | 8/8 passed |
| Repair 2 | 2 | 8/8 passed |

The number of reported errors decreased, but the final itinerary still contained violations. Some initial errors were caused by the LLM generating separate `innercity_transport` activities instead of nesting transport information within activities.

These results show that the new framework can execute repairs, but they are not yet evidence that the repair strategy consistently improves constraint satisfaction.

**Current status:** The basic framework and R0 are implemented and tested. Further verifier reliability checks are needed before final experiments.

### 3.5 Shared Verifier Reliability Improvements

After testing R0, I found three issues that could affect the reliability of our experiments. I updated `shared_verifier.py` to address them without changing the original ChinaTravel verifier or repair strategies.

**1. Commonsense verifier exceptions**

The original verifier sometimes returned Python exceptions as normal error messages, such as `KeyError: 'position'`.

I added `_split_commonsense_errors()` to separate these into `verifier_errors`, while genuine constraint violations remain in `commonsense_errors`.

**2. Silent transport translation failures**

ChinaTravel's transport translation function can silently replace failed translations with empty lists.

I added `_detect_silent_translation_failures()` to compare transport information before and after translation. It detects failures where a valid transport dictionary unexpectedly becomes an empty list.

These are recorded in `translation_errors`.

**3. Invalid itinerary structures**

The LLM sometimes generated separate `innercity_transport` activities instead of storing transport information inside the destination activity.

I added `_validate_plan_structure()` to detect unsupported activity types, missing required fields and incorrect itinerary structures.

These are recorded in `schema_errors`.

**Important:** These checks only detect and report problems. They do not automatically fix itineraries or change ChinaTravel's original constraint rules.

### 3.6 Testing After Reliability Improvements

Added `tests/test_shared_verifier_reliability.py` to test the new checks.

**Offline tests:**

- 27 new regression tests passed.
- Full test suite: 79 tests run, 0 failures and 3 errors due to missing English sandbox files.

**End-to-end R0 test:**

Reran the medium ChinaTravel case `e20241028160845920703` using GPT-4.1 with two repair attempts.

The run completed successfully, and all three verification steps were recorded.

The results confirmed that:

- Genuine constraint violations remained detectable.
- Technical errors and structural problems were recorded separately.
- The final itinerary was still marked as invalid when constraints were violated.
- R0 continued accepting repair proposals without automatically protecting previously satisfied constraints.
- Constraint-transition logging correctly identified newly introduced violations.

For example, the accommodation room-count constraint passed initially but failed after the first repair and remained violated after the second repair.

This is useful for our research because it shows how whole-plan regeneration can introduce new violations while attempting to fix existing ones.

**Current status:** The shared framework and R0 have passed the initial integration and reliability tests. The framework is ready for R1–R4 development, although the final experimental evaluation has not yet been conducted.

---

## Step 4: How to Start on Your Side

### 4.1 Setting Up

The framework is available on the `main` branch.

Follow the **Quick Setup** section in `README.md` to:

1. Clone the repository and install dependencies.
2. Download the ChinaTravel sandbox using `scripts/setup_project.py`.
3. Configure your own OpenRouter API key using `.env.example`.
4. Run one R0 test case to check your environment.

### 4.2 Main Files to Look At

Read these files in order:

| File | What to look at |
|---|---|
| `repair_strategies/base.py` | Shared repair interface |
| `repair_strategies/r0.py` | Example repair implementation |
| `shared_repair_loop.py` | How repairs are verified, accepted and logged |
| `shared_verifier.py` | Available constraint feedback |

You do not need to rewrite the original ChinaTravel planner.

### 4.3 Implementing Your Repair Strategy

Create your assigned strategy file inside `repair_strategies/`, such as `r1.py`, `r2.py`, `r3.py` or `r4.py`.

Each strategy should inherit from `BaseRepairStrategy` and implement:

```python
def repair(self, problem, raw_plan, verification_results, context, llm):
    ## Your repair logic here.

    return RepairResult(
        repaired_plan=new_plan,
        metadata={"strategy": "R1"},
    )
```

**Inputs:**

- `problem`: Original travel request and constraints.
- `raw_plan`: Current itinerary before transport translation.
- `verification_results`: Constraint feedback from the shared verifier.
- `context`: Shortlisted travel information.
- `llm`: Existing GPT-4.1 adapter.

**Output:** A `RepairResult` containing the proposed raw itinerary and metadata.

Strategies can override `accept()` if they need to reject certain repair proposals.

### 4.4 Strategy Responsibilities

| Strategy | Main task |
|---|---|
| R1 | Repair only affected itinerary slots |
| R2 | Protect previously satisfied constraints and reject harmful repairs |
| R3 | Start with targeted repair and expand the scope when needed |
| R4 | Use the constraint-interdependency matrix to guide repair |

### 4.5 Running Experiments

After your strategy has been implemented and registered in `llm_modulo.py`, you can run it using `--repair_strategy`.

Example for R0:

```bash
python run_exp.py \
  --agent LLM-modulo \
  --repair_strategy R0 \
  --llm openai/gpt-4.1 \
  --splits medium \
  --index e20241028160845920703 \
  --refine_steps 2 \
  --oracle_translation \
  --lang zh
```

splits: easy, medium subset of queries
Index : Run just a single query by its uid , default = None means to run every query in split 
Refine steps : number of repair attempts before stopping 
Oracle-translation: means the machine chable personal constraints are on 


Currently, only the new R0 is connected. R1–R4 still need to be implemented, registered and tested.

### 4.6 Things to Take Note Of

- Use the shared verifier and repair loop rather than creating a separate agent.
- Keep the itinerary in ChinaTravel's expected format. Local transport belongs inside the `transports` field of an activity.
- Use the same model, dataset and repair budget for fair comparisons.
- Rejected proposals still count toward the repair budget.
- Track both fixed constraints and newly introduced violations.
- Test your strategy using a mock LLM before making paid API calls.
- The commonsense verifier can still produce execution errors, and transport translation may fail silently. These issues need further validation before final experiments.

**For now, focus on implementing and testing your assigned repair strategy.** We can integrate and evaluate all methods together once R1–R4 are ready.


## How a query looks like 
